**Project Title**

News Article Classification using Simple RNN

**Objective**

Develop a Deep Learning model that automatically classifies news articles into predefined categories using a Simple Recurrent Neural Network (Simple RNN).

The model learns sequential relationships in textual data and predicts the correct category of unseen news articles.

**Technologies Used**

(1) Python

(2) TensorFlow / Keras

(3) Hugging Face Datasets

(4) NumPy

(5) Regular Expressions (re)

**Step 1 : Install and Load Dataset**

**Objective**

Download and load the AG News Dataset directly from Hugging Face.

**Key Goal**

(1) Load dataset without manual downloading
Separate training and testing data

In [4]:
from datasets import load_dataset

# Define the paths to the local parquet files
train_file = "/content/train-00000-of-00001.parquet"
test_file = "/content/test-00000-of-00001.parquet"

# Load the dataset from local parquet files
dataset = load_dataset("parquet", data_files={"train": train_file, "test": test_file})

# Separate train and test datasets
train_data = dataset["train"]
test_data = dataset["test"]

# Display dataset information
print(train_data)
print(test_data)

# Display one sample
print(train_data[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text', 'label', 'label_name', 'length'],
    num_rows: 120000
})
Dataset({
    features: ['text', 'label', 'label_name', 'length'],
    num_rows: 7600
})
{'text': "wall st. bears claw back into the black (reuters) reuters - short-sellers, wall street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2, 'label_name': 'Business', 'length': 144}


**Step 2 : Clean and Normalize Text**

**Objective**

Prepare clean text for neural network training.

**Key Goal**

(1) Convert all text to lowercase

(2) Remove punctuation

(3) Remove numbers

(4)Remove special characters

In [5]:
import re

# Function for cleaning text
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove numbers and punctuation
    text = re.sub(r'[^a-z\s]', '', text)

    return text

# Clean training data
train_texts = [clean_text(item["text"]) for item in train_data]

# Clean testing data
test_texts = [clean_text(item["text"]) for item in test_data]

# Labels
train_labels = [item["label"] for item in train_data]
test_labels = [item["label"] for item in test_data]

# Display sample cleaned text
print(train_texts[0])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again


**Step 3 : Tokenize Text into Integer Sequences**

**Objective**

Convert words into numerical IDs.

**Key Goal**

(1) Build vocabulary

(2) Convert words into integer tokens

(3) Prepare sequences for RNN

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Maximum vocabulary size
vocab_size = 10000

# Create tokenizer
tokenizer = Tokenizer(num_words=vocab_size,
                      oov_token="<OOV>")

# Learn vocabulary
tokenizer.fit_on_texts(train_texts)

# Convert sentences into sequences
X_train = tokenizer.texts_to_sequences(train_texts)
X_test = tokenizer.texts_to_sequences(test_texts)

# Display one sequence
print(X_train[0])

[392, 325, 1526, 1, 100, 55, 2, 813, 24, 24, 1, 392, 1989, 1, 5, 1, 35, 3894, 738, 296]


**Step 4 : Pad and Truncate Sequences**

**Objective**

Ensure all news articles have equal sequence length.

**Key Goal**

(1) Uniform input size

(2) Easier batch processing

(3) Faster model training

In [7]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Maximum sequence length
max_length = 50

# Pad training data
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

# Pad testing data
X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print(X_train.shape)

(120000, 50)


**Step 5 : One-Hot Encode Labels**

**Objective**

Convert category labels into categorical vectors.

**Key Goal**

(1) Convert labels into probability format

(2) Compatible with softmax output layer

In [8]:
from tensorflow.keras.utils import to_categorical

# Number of news categories
num_classes = 4

# One-hot encoding
y_train = to_categorical(train_labels,
                         num_classes)

y_test = to_categorical(test_labels,
                        num_classes)

print(y_train[0])

[0. 0. 1. 0.]


**Step 6 : Build the Simple RNN Model**

**Objective**

Design a neural network capable of learning sequential text patterns.

**Key Goal**

(1) Learn word embeddings

(2) Capture contextual information

(3) Predict news category

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense

# Create Sequential model
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

# Simple RNN Layer
model.add(
    SimpleRNN(64)
)

# Output Layer
model.add(
    Dense(
        num_classes,
        activation="softmax"
    )
)

# Display architecture
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Step 7 : Compile and Train the Model**

**Objective**

Train the neural network to classify news articles.

**Key Goal**

(1) Learn from training data

(2) Optimize weights

(3) Validate performance

In [11]:
# Compile model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train model
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - accuracy: 0.8971 - loss: 0.3194 - val_accuracy: 0.8810 - val_loss: 0.3646
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 36s 24ms/step - accuracy: 0.9157 - loss: 0.2629 - val_accuracy: 0.8671 - val_loss: 0.3828
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 38s 25ms/step - accuracy: 0.9275 - loss: 0.2271 - val_accuracy: 0.8810 - val_loss: 0.3631
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.9359 - loss: 0.1994 - val_accuracy: 0.8651 - val_loss: 0.4258
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.9434 - loss: 0.1760 - val_accuracy: 0.8784 - val_loss: 0.3962


**Step 8 : Evaluate the Model**

**Objective**

Measure model performance on unseen test data.

**Key Goal**

(1) Calculate accuracy

(2) Evaluate generalization ability

In [12]:
# Evaluate on test dataset
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8893 - loss: 0.3673
Test Loss : 0.3672622740268707
Test Accuracy : 0.8893421292304993


**Step 9 : Predict a Custom News Article**

**Objective**

Classify a user-provided news headline.

**Key Goal**

(1) Demonstrate real-world prediction

(2) Validate trained model

In [13]:
# Category names
classes = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

# Sample news
news = "NASA launches a new satellite to study climate change."

# Clean
news = clean_text(news)

# Tokenize
sequence = tokenizer.texts_to_sequences([news])

# Pad
sequence = pad_sequences(
    sequence,
    maxlen=max_length,
    padding="post"
)

# Predict
prediction = model.predict(sequence)

# Get class index
predicted_class = prediction.argmax()

print("Predicted Category :", classes[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 514ms/step
Predicted Category : Science/Technology


**Overall Project Objectives**

(1) Build a Deep Learning model for text classification.

(2) Learn how RNNs process sequential textual data.

(3) Convert raw text into machine-readable numerical sequences.

(4) Train a Simple RNN to classify news articles.

(5) Evaluate classification accuracy on unseen data.

(6) Demonstrate real-world prediction using custom news headlines.


**Key Goal Achievements**

(1) Loaded the AG News Dataset directly from Hugging Face.

(2) Cleaned and normalized raw news text.

(3) Converted words into integer tokens using Keras Tokenizer.

(4) Standardized sequence lengths using padding and truncation.

(5) One-hot encoded categorical labels for multi-class classification.

(6) Built a Simple RNN architecture with Embedding, SimpleRNN, and Dense layers.

(7) Compiled and trained the model using the Adam optimizer and categorical cross-entropy loss.

(8) Evaluated the model on unseen test data using accuracy and loss metrics.

(9) Performed inference on a custom news article to demonstrate practical deployment.

**Expected Performance**

With the above configuration (Embedding → SimpleRNN → Dense, 5 epochs), a Simple RNN on the AG News dataset typically achieves around 85–90% test accuracy. Exact results vary depending on random initialization, preprocessing, hyperparameters, and the execution environment.